# 2B · Conditions, Loops & Functions
### Financial Analytics — Module 2

In 2A you learned to *store* financial data. Now you learn to *act* on it — the three ideas that turn data into analysis:

1. **Conditions** — make a decision (`if`)
2. **Loops** — repeat over many items (`for`)
3. **Functions** — package logic to reuse (`def`)

Every backtest, every risk report, every scorecard is these three, combined.

---
## 1. Conditions: `if` / `elif` / `else`

The indentation is not decoration — it's how Python knows what belongs inside the `if`.

In [ ]:
price = 3450.75
stop_loss = 3300.00
target = 3600.00

if price <= stop_loss:
    action = "SELL - stop loss hit"
elif price >= target:
    action = "SELL - target reached"
else:
    action = "HOLD"

print(f"Price Rs {price:,.2f} -> {action}")

**Try it:** change `price` to 3250 and re-run. Then to 3650. Watch the decision change.

In [ ]:
# A risk-limit check - the everyday shape of banking and markets code
position_value = 5_200_000     # Rs 52 lakh  (underscores are just readability)
limit = 5_000_000

if position_value > limit:
    breach = position_value - limit
    print(f"BREACH: over limit by Rs {breach:,.0f}")
else:
    headroom = limit - position_value
    print(f"OK: Rs {headroom:,.0f} of headroom remaining")

### ✏️ Exercise 1
Write a rule that classifies a client by AUM: below Rs 10 lakh -> "Mass", Rs 10 lakh to 1 crore -> "Affluent", above Rs 1 crore -> "HNI". Print the segment for `aum = 4_500_000`.

In [ ]:
aum = 4_500_000

# your code here


---
## 2. Loops: `for`

A loop repeats the same work across many items. This is where finance code earns its keep — 1 stock or 5,000, the code is identical.

In [ ]:
watchlist = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS"]

for ticker in watchlist:
    print("Checking", ticker)

In [ ]:
# Loop over a dict: the portfolio valuation pattern
holdings = {"TCS.NS": 95, "HDFCBANK.NS": 200, "INFY.NS": 120, "ITC.NS": 500}
prices   = {"TCS.NS": 3450.75, "HDFCBANK.NS": 1520.40, "INFY.NS": 1480.50, "ITC.NS": 415.20}

total = 0
for ticker, qty in holdings.items():
    value = qty * prices[ticker]
    total = total + value                      # accumulate
    print(f"{ticker:<14} {qty:>5} x Rs {prices[ticker]:>9,.2f} = Rs {value:>12,.2f}")

print("-" * 52)
print(f"{'TOTAL':<14} {'':>5}   {'':>12}   Rs {total:>12,.2f}")

**Read that output.** You just valued a portfolio — the single most common calculation in asset management. The loop did in 5 lines what would be a spreadsheet of formulas.

In [ ]:
# Loop with a condition inside - the shape of EVERY trading strategy
closes = [3450.75, 3462.10, 3438.90, 3475.25, 3491.00, 3468.30, 3502.15]
buy_below = 3450.00

signals = []
for day, price in enumerate(closes):          # enumerate gives you position AND value
    if price < buy_below:
        signals.append((day, price, "BUY"))

print("Buy signals:", signals)
print(f"{len(signals)} signal(s) out of {len(closes)} days")

### ✏️ Exercise 2
Loop over `closes` and count how many days closed **higher** than the previous day. (Hint: start from index 1 and compare `closes[i]` with `closes[i-1]`.)

In [ ]:
closes = [3450.75, 3462.10, 3438.90, 3475.25, 3491.00, 3468.30, 3502.15]

up_days = 0
# your code here

print("Up days:", up_days)

# CHECK: the answer should be 4

---
## 3. Functions: `def`

A function is a named, reusable piece of logic. Write it once, trust it forever.

**Why this matters in finance:** if your return calculation lives in one function, you fix a bug in one place. If it's copy-pasted across 40 cells, you have 40 bugs.

In [ ]:
def simple_return(buy_price, sell_price):
    """Return the percentage gain/loss on a trade."""
    return (sell_price - buy_price) / buy_price * 100


# Now use it as many times as you like
print(round(simple_return(3450.75, 3502.15), 2), "%")
print(round(simple_return(1480.50, 1421.30), 2), "%")

In [ ]:
def portfolio_value(holdings, prices):
    """Total value of a holdings dict, priced with a prices dict."""
    total = 0
    for ticker, qty in holdings.items():
        total += qty * prices[ticker]        # += is shorthand for total = total + ...
    return total


holdings = {"TCS.NS": 95, "HDFCBANK.NS": 200, "INFY.NS": 120}
prices   = {"TCS.NS": 3450.75, "HDFCBANK.NS": 1520.40, "INFY.NS": 1480.50}

print(f"Portfolio: Rs {portfolio_value(holdings, prices):,.2f}")

# TRACER BULLET (Module 0's verify habit): test on a case you can check by hand
tiny = portfolio_value({"X": 10}, {"X": 100})
print("Tracer test - expect 1000, got:", tiny)

**That tracer-bullet habit is the whole difference between code that works and code you *trust*.** Always test a function on an input whose answer you already know.

In [ ]:
# Default arguments: sensible defaults, override when needed
def cagr(start_value, end_value, years=1):
    """Compound annual growth rate, as a percentage."""
    return ((end_value / start_value) ** (1 / years) - 1) * 100


print("1 year:", round(cagr(100, 112), 2), "%")
print("5 years:", round(cagr(100, 180, years=5), 2), "%")

### ✏️ Exercise 3
Write `brokerage_cost(trade_value, rate_pct=0.03, minimum=20)` that returns the brokerage: `trade_value * rate_pct / 100`, but never less than `minimum`. Test it on a Rs 500 trade (should return 20) and a Rs 200,000 trade (should return 60).

In [ ]:
def brokerage_cost(trade_value, rate_pct=0.03, minimum=20):
    # your code here
    pass


print(brokerage_cost(500))       # expect 20
print(brokerage_cost(200_000))   # expect 60.0

---
## 4. List comprehensions — the shortcut you'll see everywhere

A compact way to build a list from another list. Same result as a loop, one line.

In [ ]:
closes = [3450.75, 3462.10, 3438.90, 3475.25, 3491.00]

# The long way
rounded_long = []
for p in closes:
    rounded_long.append(round(p))

# The comprehension way - identical result
rounded = [round(p) for p in closes]
print(rounded)

# With a filter attached
above_3460 = [p for p in closes if p > 3460]
print("Above 3460:", above_3460)

---
## Recap

| Idea | Syntax | Finance use |
|---|---|---|
| Condition | `if / elif / else` | Trading rules, risk limits, client segmentation |
| Loop | `for x in items:` | Valuing every holding, scanning every trading day |
| Function | `def name(args): return ...` | Return calcs, valuations, any repeated logic |
| Comprehension | `[f(x) for x in items]` | Compact transforms and filters |

**Next:** notebook 2C — combine all of it into a working position tracker.

---
*AI disclosure: did AI help you here? Note what for: ______*